# Lecture 9 Demo 2: pandas in Practice

Aligned with the **pandas in Practice** section of Lecture 9.

## What this notebook covers
- inspect a new table before transforming it
- filter rows and columns to answer a question
- derive safe feature columns with `.loc`
- parse a messy text column with vectorized code
- build a summary table from many rows
- join lookup tables, handle missing values explicitly, and parse time columns safely

In this notebook each section answers a concrete dataframe task.


In [13]:
from __future__ import annotations

import pandas as pd

from demo_utils import data_path, section

section("Versions")
print(f"pandas: {pd.__version__}")
print("This notebook is organized around typical pandas workflows.")



Versions
pandas: 3.0.2
This notebook is organized around typical pandas workflows.


## 1. Inspect the table before touching it

We use local files so the notebook is reliable offline.

As you run the next cell, ask:
- what does one row represent?
- which columns are numeric, text, or boolean?
- where are the missing values?
- which columns will likely matter later?


In [14]:
section("Inspect the table before touching it")

missions = pd.read_csv(data_path("space_missions.csv"), dtype_backend="pyarrow")
quakes = pd.read_csv(data_path("earthquakes.csv"), dtype_backend="pyarrow")

print("First rows:")
print(missions.head(5).to_string(index=False))
print("\nMissions dtypes:")
print(missions.dtypes.to_string())
print("\nMissing values:")
print(missions.isna().sum().to_string())



Inspect the table before touching it
First rows:
    mission_name         agency  launch_year  duration_days  crew_size        destination success  distance_km  cost_million_usd
       Apollo 11           NASA         1969            8.0          3               Moon    True       384400           25400.0
       Voyager 1           NASA         1977        17167.0          0 Interstellar Space    True  23500000000             865.0
 Mars Pathfinder           NASA         1996          267.0          0               Mars    True    225000000             280.0
ISS Expedition 1 NASA/Roscosmos         2000          136.0          3    Low Earth Orbit    True          408          150000.0
   SpaceX Demo-2         SpaceX         2020           64.0          2                ISS    True          408            1500.0

Missions dtypes:
mission_name        string[pyarrow]
agency              string[pyarrow]
launch_year          int64[pyarrow]
duration_days       double[pyarrow]
crew_size     

## 2. Filter the relevant rows, then derive useful labels

A typical pandas question is not "show me the whole dataframe" but "show me the rows and columns that answer this question."

The next cell does two common tasks:
1. filter the mission table to answer a Moon-missions question
2. derive `mission_style` and `distance_band` so later summaries are easier to read

Notice how `.loc[rows, columns]` keeps the filtering logic readable, and how explicit column assignment makes derived features easy to audit.


In [15]:
section("Filter to the relevant rows")

moon_missions = missions.loc[
    missions["destination"] == "Moon",
    ["mission_name", "launch_year", "agency", "cost_million_usd"],
].sort_values("launch_year")

print(moon_missions.to_string(index=False))

section("Derive meaningful columns safely")

missions = missions.copy()
missions["mission_style"] = "uncrewed"
missions.loc[missions["crew_size"] > 0, "mission_style"] = "crewed"

missions["distance_band"] = "local"
missions.loc[missions["distance_km"] >= 384_400, "distance_band"] = "lunar_or_beyond"
missions.loc[missions["distance_km"] >= 1_000_000, "distance_band"] = "deep_space"

print(
    missions[
        ["mission_name", "crew_size", "mission_style", "distance_km", "distance_band"]
    ].head(8).to_string(index=False)
)



Filter to the relevant rows
 mission_name  launch_year agency  cost_million_usd
     Apollo 8         1968   NASA            2500.0
    Apollo 11         1969   NASA           25400.0
    Apollo 13         1970   NASA           25400.0
    Chang'e 4         2018   CNSA             170.0
Chandrayaan-2         2019   ISRO             140.0
    Chang'e 5         2020   CNSA             200.0
    Artemis I         2022   NASA            4100.0
Chandrayaan-3         2023   ISRO              75.0
   Artemis II         2026   NASA            4500.0

Derive meaningful columns safely
    mission_name  crew_size mission_style  distance_km   distance_band
       Apollo 11          3        crewed       384400 lunar_or_beyond
       Voyager 1          0      uncrewed  23500000000      deep_space
 Mars Pathfinder          0      uncrewed    225000000      deep_space
ISS Expedition 1          3        crewed          408           local
   SpaceX Demo-2          2        crewed          408        

## 3. Parse messy text and end with a summary table

The next cell combines two practical dataframe moves:
- extract structured numeric values from a text-heavy column
- compress many detailed mission rows into a small summary table

Try to read each pipeline top to bottom like a recipe: filter, derive, group, aggregate, sort.


In [18]:
section("Parse a messy text column")

wkt_pattern = r"POINT Z \((?P<longitude>.*?) (?P<latitude>.*?) (?P<depth>.*?)\)"

coordinates = (
    quakes["geometry"]
    .str.extract(wkt_pattern)
    .astype("float32[pyarrow]")
)

quakes = pd.concat([quakes, coordinates], axis=1)

print(
    quakes[["place", "latitude", "longitude", "depth"]]
    .head(5)
    .to_string(index=False)
)

section("Build a summary table")

destination_summary = (
    missions[missions["destination"].isin(["Moon", "Mars", "ISS"])]
    .assign(
        cost_per_day=lambda df: (df["cost_million_usd"] / df["duration_days"]).round(2)
    )
    .groupby("destination")
    .agg(
        mission_count=("mission_name", "size"),
        avg_cost_per_day=("cost_per_day", "mean"),
        crewed_missions=("mission_style", lambda s: (s == "crewed").sum()),
    )
    .sort_values("avg_cost_per_day", ascending=False)
    .round(2)
)

print(destination_summary.to_string())



Parse a messy text column
                             place   latitude   longitude      depth
      87 km SW of Nikolski, Alaska  52.333801 -169.676605       10.0
37 km WSW of Asadābād, Afghanistan    34.7061   70.793404        8.0
98 km E of Severo-Kuril’sk, Russia  50.823601  157.496506  53.408001
         east of the Kuril Islands  49.391899  160.036102       10.0
 81 km SW of Acajutla, El Salvador    13.0753  -90.360497       10.0

Build a summary table
             mission_count  avg_cost_per_day  crewed_missions
destination                                                  
Moon                     9             937.9                4
ISS                      1             23.44                1
Mars                     9              0.83                0


## 4. Three more starter patterns worth knowing

These three patterns are worth learning early because they show up in real work all the time:
- attach metadata from another table with `merge(..., validate="many_to_one")`
- treat missing values differently depending on what the column means
- parse time columns once, then use `.dt` features for grouping


In [ ]:
section("Attach lookup data with merge()")

agency_info = pd.DataFrame(
    {
        "agency": ["NASA", "CNSA", "ISRO"],
        "region": ["USA", "China", "India"],
        "crew_program": ["yes", "planned", "planned"],
    }
)

moon_with_agency = moon_missions.merge(
    agency_info,
    on="agency",
    how="left",
    validate="many_to_one",
)

print(moon_with_agency.to_string(index=False))

section("Handle missing values explicitly")

quake_alerts = (
    quakes.loc[:, ["place", "mag", "mmi", "alert"]]
    .assign(alert=lambda df: df["alert"].fillna("unknown"))
    .dropna(subset=["mmi"])
    .sort_values(["mmi", "mag"], ascending=[False, False])
    .head(5)
)

print(quake_alerts.to_string(index=False))

section("Parse time columns before grouping")

quakes = quakes.assign(
    time_utc=pd.to_datetime(quakes["time"], unit="ms", utc=True),
)

quakes_by_year = (
    quakes.assign(year=quakes["time_utc"].dt.year)
    .groupby("year")
    .agg(
        event_count=("id", "size"),
        max_mag=("mag", "max"),
    )
    .sort_index()
    .tail(6)
)

print(quakes_by_year.to_string())


## Wrap-up prompt

1. Which cell helped you understand the table before analysis started?
2. Which part used `.loc` to make feature engineering explicit?
3. Which section attached metadata with `merge(..., validate="many_to_one")`?
4. Which section handled missing `alert` and `mmi` differently?
5. Which section parsed timestamps with `pd.to_datetime(..., utc=True)`?
6. Which pipeline ended with a compact answer table?

If you can answer those questions, you are learning pandas as a workflow, not as a bag of methods.
